In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter
import pandas as pd


from bragg_onion.materials import MaterialFileSpec, ConstantDispersion, load_materials
from bragg_onion.sweep import (
    run_bragg_onion_sweep,
    make_plotting_aliases,
    plot_sweep_metric,
    plot_sweep_heatmap,
    plot_sweep_colour_strip,
)

from bragg_onion.scattering_analysis import (
    plot_differential_scattering_vs_angle,
    plot_polar_scattering,
    plot_forward_side_backward_vs_wavelength,
    plot_partition_pie_at_wavelength,
    print_scattering_metrics,
    scattering_metrics_dataframe,
)

from bragg_onion.colour_solid_plotting import (
    load_rosch_macadam_colour_solid,
    plot_rosch_macadam_colour_solid,
    add_cielab_point,
)

import bragg_onion.fields as bo_fields

from bragg_onion.fields import (
    compute_field_map,
    plot_field_magnitude,
    plot_poynting_streamlines,
    plot_poynting_vectors,
    make_line_seeds,
    make_circle_seeds,
    make_edge_seeds,
)

In [ ]:
# Helper to reconstruct geometry / stack for the selected case
# This is useful because SweepCaseResult stores the scattering result, but near-field calculations often need the stack again.
# This helper assumes your sweep was run in "peak_wavelength" mode and that your notebook still has:

# mat_A
# mat_B
# FIXED_GEOMETRY_KWARGS

# already defined.
from bragg_onion.geometry import build_bragg_onion_from_peak_wavelength, resolve_layer_stack

def rebuild_stack_from_selected_row(
    row_sel,
    *,
    material_a,
    material_b,
    fixed_geometry_kwargs=None,
    extinction_modifiers=None,
    extra_outer_shell=None,
):
    """
    Reconstruct geometry + resolved stack for one selected sweep row.
    """
    fixed_geometry_kwargs = dict(fixed_geometry_kwargs or {})

    geom_kwargs = dict(fixed_geometry_kwargs)
    geom_kwargs.update(
        {
            "peak_wavelength_m": float(row_sel["peak_wavelength_m"]),
            "n_layers": int(row_sel["n_layers"]),
            "outer_layer": str(row_sel["outer_layer"]),
            "core_thickness_factor": float(row_sel["core_thickness_factor"]),
        }
    )

    geometry = build_bragg_onion_from_peak_wavelength(
        material_a=material_a,
        material_b=material_b,
        extra_outer_shell=extra_outer_shell,
        **geom_kwargs,
    )

    stack = resolve_layer_stack(
        geometry=geometry,
        material_a=material_a,
        material_b=material_b,
        extinction_modifiers=extinction_modifiers,
    )

    return geometry, stack

In [ ]:
def set_publication_style():
    plt.rcParams.update({
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 9,
        "legend.fontsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "lines.linewidth": 1.6,
        "figure.figsize": (6.0, 4.0),
        "savefig.bbox": "tight",
    })


def save_figure_multi(
    base_path: Path,
    *,
    fig=None,
    save_pdf: bool = True,
    save_png: bool = False,
    dpi: int = 600,
    show: bool = True,
    close: bool = False,
):
    if fig is None:
        fig = plt.gcf()

    base_path = Path(base_path)
    base_path.parent.mkdir(parents=True, exist_ok=True)

    if save_pdf:
        fig.savefig(base_path.with_suffix(".pdf"))
    if save_png:
        fig.savefig(base_path.with_suffix(".png"), dpi=dpi)

    if show:
        plt.show()
    if close:
        plt.close(fig)

In [ ]:
def get_top_n_cases(
    df_plot: pd.DataFrame,
    *,
    n: int = 5,
    sort_by=("eta_C", "q_collected_geom_max"),
    ascending=(False, False),
):
    return df_plot.sort_values(list(sort_by), ascending=list(ascending)).head(n).copy()


def get_cases_by_index(df_plot: pd.DataFrame, case_indices):
    case_indices = list(case_indices)
    return df_plot[df_plot["case_index"].isin(case_indices)].copy()


def get_cases_by_filters(df_plot: pd.DataFrame, filters: list[dict]):
    rows = []
    for filt in filters:
        mask = np.ones(len(df_plot), dtype=bool)

        for key, val in filt.items():
            if key not in df_plot.columns:
                raise KeyError(f"Column '{key}' not found in df_plot.")
            if isinstance(val, float):
                mask &= np.isclose(df_plot[key].to_numpy(dtype=float), val, atol=1e-6)
            else:
                mask &= df_plot[key].astype(str).to_numpy() == str(val)

        subset = df_plot.loc[mask]
        if subset.empty:
            print(f"No match for filter: {filt}")
        else:
            rows.append(subset.iloc[0])

    if not rows:
        raise ValueError("No cases matched the requested filters.")

    return pd.DataFrame(rows)

In [ ]:
def make_case_folder_name(row_sel, *, system_label: str | None = None) -> str:
    parts = []
    if system_label:
        parts.append(system_label.replace(" ", "_").replace("/", "_"))

    parts.extend([
        f"case_{int(row_sel['case_index']):03d}",
        f"lam_{float(row_sel['design_peak_wavelength_nm']):.0f}nm",
        f"layers_{int(row_sel['n_layers'])}",
        f"outer_{str(row_sel['outer_layer'])}",
    ])

    return "_".join(parts)

In [ ]:
def analyze_case_part3_report(
    row_sel,
    sweep_result,
    *,
    material_a,
    material_b,
    medium,
    fixed_geometry_kwargs=None,
    alpha_deg=30.0,
    field_plane="xz",
    field_npts=401,
    field_extent_factor=2.5,
    save_root=None,
    system_label: str | None = None,
    save_pdf=True,
    save_png=False,
    show=True,
):
    set_publication_style()

    case_idx = int(row_sel["case_index"])
    case = next(c for c in sweep_result.case_results if c.case_index == case_idx)

    scat = case.scattering_result
    integ = case.integrated_result

    # --------------------------------------------------------
    # Build case directory
    # --------------------------------------------------------
    case_dir = None
    if save_root is not None:
        case_dir = Path(save_root) / make_case_folder_name(row_sel, system_label=system_label)
        case_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------
    metrics = scattering_metrics_dataframe(
        scat,
        design_wavelength_nm=row_sel["design_peak_wavelength_nm"],
        alpha_deg=alpha_deg,
    )
    display(metrics)

    if case_dir is not None:
        metrics.to_csv(case_dir / "metrics.csv", index=False)

    # --------------------------------------------------------
    # Differential scattering
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_differential_scattering_vs_angle(
        scat,
        wavelengths_nm=[450, 500, row_sel["design_peak_wavelength_nm"], 600, 650],
        normalization="geom",
        ax=ax,
    )
    ax.set_title("Differential scattering vs angle")
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "diff_scattering", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    # --------------------------------------------------------
    # Forward / side / backward spectra
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_forward_side_backward_vs_wavelength(
        scat,
        alpha_deg=alpha_deg,
        normalize="geom",
        ax=ax,
    )
    ax.set_title(f"Forward / side / backward efficiencies (α = {alpha_deg:.1f}°)")
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "partition_spectra", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    # --------------------------------------------------------
    # Pie chart
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_partition_pie_at_wavelength(
        scat,
        alpha_deg=alpha_deg,
        design_wavelength_nm=row_sel["design_peak_wavelength_nm"],
        ax=ax,
    )
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "partition_pie", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    # --------------------------------------------------------
    # Rebuild geometry / stack for field maps
    # --------------------------------------------------------
    geometry, stack = rebuild_stack_from_selected_row(
        row_sel,
        material_a=material_a,
        material_b=material_b,
        fixed_geometry_kwargs=fixed_geometry_kwargs,
    )

    field_map = compute_field_map(
        stack=stack,
        wavelength_m=row_sel["design_peak_wavelength_nm"] * 1e-9,
        n_medium=medium,
        plane=field_plane,
        npts=field_npts,
        extent_outer_radius_factor=field_extent_factor,
    )

    # --------------------------------------------------------
    # E-field magnitude
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_field_magnitude(
        field_map,
        quantity="E",
        kind="scattered",
        scale="db",
        ax=ax,
    )
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "field_E_scattered", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    # --------------------------------------------------------
    # Delta-flow streamlines (edge-seeded, longer)
    # --------------------------------------------------------
    edge_seeds_nm = make_edge_seeds(
        field_map,
        edge="min_coord2",
        n_seeds=35,
        margin_nm=5.0,
        start_fraction=0.10,
        end_fraction=0.90,
    )

    fig, ax = plt.subplots()
    plot_poynting_streamlines(
        field_map,
        flow_kind="delta",
        background_quantity="S",
        background_kind="delta",
        background_scale="db",
        background_clip_percentile_high=99.7,
        streamline_density=4.0,
        streamline_color="white",
        streamline_linewidth=0.8,
        start_points_nm=edge_seeds_nm,
        normalize_vectors=True,
        min_speed_fraction=0.001,
        mask_inside_sphere=False,
        integration_direction="forward",
        minlength=0.05,
        maxlength=20.0,
        broken_streamlines=False,
        arrowstyle="simple,head_length=1.0,head_width=0.35,tail_width=0.15",
        arrowsize=1.3,
        ax=ax,
    )
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "poynting_delta_streamlines", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    # --------------------------------------------------------
    # Delta-flow vectors
    # --------------------------------------------------------
    fig, ax = plt.subplots()
    plot_poynting_vectors(
        field_map,
        flow_kind="delta",
        background_quantity="S",
        background_kind="delta",
        background_scale="db",
        background_clip_percentile_high=99.7,
        step=12,
        normalize_vectors=True,
        min_speed_fraction=0.001,
        mask_inside_sphere=False,
        vector_color="white",
        vector_scale=18,
        vector_width=0.003,
        pivot="mid",
        quiver_headwidth=2.6,
        quiver_headlength=4.8,
        quiver_headaxislength=4.8,
        ax=ax,
    )
    plt.tight_layout()

    if case_dir is not None:
        save_figure_multi(case_dir / "poynting_delta_vectors", fig=fig, save_pdf=save_pdf, save_png=save_png, show=show, close=False)
    elif show:
        plt.show()

    return metrics

In [ ]:
TOP_N = 5
top_cases = get_top_n_cases(
    df_plot,
    n=TOP_N,
    sort_by=("eta_C", "q_collected_geom_max"),
    ascending=(False, False),
)

REPORT_ROOT = OUTPUT_DIR / "part3_top_cases_reports"

for _, row in top_cases.iterrows():
    analyze_case_part3_report(
        row,
        sweep_result,
        material_a=mat_A,
        material_b=mat_B,
        medium=medium,
        fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
        alpha_deg=alpha_deg,
        save_root=REPORT_ROOT,
        system_label="PS_P2VP",
        save_pdf=True,
        save_png=False,
        show=False,
    )

In [ ]:
# Cell 6 — run polished reports for specific selected cases
# By case index
selected_cases = get_cases_by_index(df_plot, [0, 5, 12])

for _, row in selected_cases.iterrows():
    analyze_case_part3_report(
        row,
        sweep_result,
        material_a=mat_A,
        material_b=mat_B,
        medium=medium,
        fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
        alpha_deg=alpha_deg,
        save_root=OUTPUT_DIR / "part3_selected_cases_reports",
        system_label="PS_P2VP",
        save_pdf=True,
        save_png=False,
        show=True,
    )

# By filters
selected_cases = get_cases_by_filters(
    df_plot,
    [
        {"design_peak_wavelength_nm": 550.0, "n_layers": 20, "outer_layer": "A"},
        {"design_peak_wavelength_nm": 650.0, "n_layers": 50, "outer_layer": "A"},
    ],
)

for _, row in selected_cases.iterrows():
    analyze_case_part3_report(
        row,
        sweep_result,
        material_a=mat_A,
        material_b=mat_B,
        medium=medium,
        fixed_geometry_kwargs=FIXED_GEOMETRY_KWARGS,
        alpha_deg=alpha_deg,
        save_root=OUTPUT_DIR / "part3_filtered_cases_reports",
        system_label="PS_P2VP",
        save_pdf=True,
        save_png=False,
        show=True,
    )    

# Part C — optional helper to compare PS/P2VP with another material system later

In [ ]:
def labelled_sweep_dataframe(sweep_result, label: str) -> pd.DataFrame:
    df = make_plotting_aliases(sweep_result.dataframe).copy()
    df["material_system"] = label
    return df

In [ ]:
# Example use
# df_ps_p2vp = labelled_sweep_dataframe(sweep_result_ps_p2vp, "PS/P2VP")
# df_other = labelled_sweep_dataframe(sweep_result_other, "Other system")
# df_compare = pd.concat([df_ps_p2vp, df_other], ignore_index=True)

In [ ]:
# Comparison plot helper
def compare_metric_between_systems(
    df_compare: pd.DataFrame,
    *,
    metric: str,
    x: str = "design_peak_wavelength_nm",
    n_layers: int | None = None,
    outer_layer: str | None = None,
):
    df = df_compare.copy()

    if n_layers is not None:
        df = df[df["n_layers"] == n_layers]
    if outer_layer is not None:
        df = df[df["outer_layer"] == outer_layer]

    fig, ax = plt.subplots(figsize=(6, 4))

    for system_name, grp in df.groupby("material_system"):
        grp = grp.sort_values(x)
        ax.plot(grp[x], grp[metric], marker="o", label=system_name)

    ax.set_xlabel(x)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} comparison")
    ax.legend()
    plt.tight_layout()
    plt.show()